In [1]:
import pandas as pd
from src.utils.paths import load_paths

paths = load_paths()

vnat_feats = pd.read_parquet(paths.data_processed / "vnat" / "features_trainable.parquet")
iscx_feats = pd.read_parquet(paths.data_processed / "iscx" / "features.parquet")

iscx_feats = iscx_feats[iscx_feats["q_min_packets_ok"] == 1.0].copy()

df_all = pd.concat([vnat_feats, iscx_feats], ignore_index=True)

print("Splits:")
print(df_all["split"].value_counts())
print("\nLabels:")
print(df_all["label"].value_counts())

Splits:
split
train        11306
iscx_val       985
iscx_test      341
test           210
val            147
Name: count, dtype: int64

Labels:
label
0    9672
1    3317
Name: count, dtype: int64


In [2]:
from src.models.xgb_train import train_xgboost
from src.utils.logging import setup_logger

logger = setup_logger(level="INFO")
xgb_yaml = paths.configs_dir / "xgb.yaml"

res = train_xgboost(paths=paths, xgb_yaml=xgb_yaml, df=df_all)

print("Saved model:", res.model_path)
print("Saved metrics:", res.metrics_path)
print("Saved preds:", res.preds_path)

print("\nTrain:", res.metrics["splits"]["train"])
print("\nVal (VNAT only):", res.metrics["splits"]["val"])
print("\nVNAT Test:", res.metrics["splits"]["test"])
print("\nFirewall policy:", res.metrics.get("firewall_policy", res.metrics.get("policy_thresholds", {})))


DEBUG mean/std (first 5):
[-0.318 -0.753 -0.76  -0.7   -0.479]
[0.    0.029 0.061 0.055 0.   ]
[0]	train-logloss:0.64991	train-auc:0.98884	train-aucpr:0.95329	val-logloss:0.65555	val-auc:0.89630	val-aucpr:0.36948
[100]	train-logloss:0.01584	train-auc:0.99998	train-aucpr:0.99993	val-logloss:0.05972	val-auc:0.99877	val-aucpr:0.98608
[200]	train-logloss:0.00360	train-auc:1.00000	train-aucpr:1.00000	val-logloss:0.04427	val-auc:1.00000	val-aucpr:1.00000
[300]	train-logloss:0.00174	train-auc:1.00000	train-aucpr:1.00000	val-logloss:0.04645	val-auc:1.00000	val-aucpr:1.00000
[312]	train-logloss:0.00165	train-auc:1.00000	train-aucpr:1.00000	val-logloss:0.04567	val-auc:1.00000	val-aucpr:1.00000
Saved model: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\xgb\model.json
Saved metrics: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\xgb\metrics.json
Saved preds: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\xgb\preds.parquet

Train: {'n': 11306, 'pos': 2378, 'neg': 8928

In [3]:
import json
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

from src.pipeline.artifacts import default_feature_artifacts
from src.pipeline.feature_pipeline import FeaturePipeline

model_path = paths.repo_root / "artifacts" / "xgb" / "model.json"
booster = xgb.Booster()
booster.load_model(str(model_path))

feature_art = default_feature_artifacts(paths.artifacts_dir / "features")
pipeline = FeaturePipeline.load(feature_art)

X_all = pipeline.transform(df_all)
feat_cols = pipeline.model_feature_names()



iscx_test_df = df_all[df_all["split"] == "iscx_test"].copy()
X_iscx = X_all.loc[iscx_test_df.index, feat_cols].to_numpy(dtype=float)
y_iscx = iscx_test_df["label"].to_numpy(dtype=int)

p_iscx = booster.predict(xgb.DMatrix(X_iscx, feature_names=feat_cols))

print("ISCX TEST ROC:", roc_auc_score(y_iscx, p_iscx))
print("ISCX TEST PR :", average_precision_score(y_iscx, p_iscx))

metrics_path = paths.repo_root / "artifacts" / "xgb" / "metrics.json"
m = json.loads(metrics_path.read_text(encoding="utf-8"))

pol = m.get("policy_thresholds", {})
if "val_fpr_0_1pct" in pol:
    thr = float(pol["val_fpr_0_1pct"]["threshold"])
    pol_name = "val_fpr_0_1pct"
else:
    thr = float(pol["val_fpr_1pct"]["threshold"])
    pol_name = "val_fpr_1pct"

print("Using firewall policy:", pol_name, "threshold:", thr)

yhat = (p_iscx >= thr).astype(int)
tn, fp, fn, tp = confusion_matrix(y_iscx, yhat).ravel()

prec = tp / (tp + fp + 1e-9)
rec  = tp / (tp + fn + 1e-9)
fpr  = fp / (fp + tn + 1e-9)

print("ISCX TEST @ firewall thr:", {"tn":tn,"fp":fp,"fn":fn,"tp":tp, "precision":prec, "recall":rec, "fpr":fpr})

ISCX TEST ROC: 0.8968956514644693
ISCX TEST PR : 0.9623609764424705
Using firewall policy: val_fpr_0_1pct threshold: 0.8754015564918518
ISCX TEST @ firewall thr: {'tn': np.int64(102), 'fp': np.int64(1), 'fn': np.int64(81), 'tp': np.int64(157), 'precision': np.float64(0.9936708860696604), 'recall': np.float64(0.6596638655434468), 'fpr': np.float64(0.00970873786398341)}


In [7]:
vnat_test_df = df_all[df_all["split"] == "test"].copy()
X_vnat_test = X_all.loc[vnat_test_df.index, feat_cols].to_numpy(dtype=float)
y_vnat_test = vnat_test_df["label"].to_numpy(dtype=int)
p_vnat_test = booster.predict(xgb.DMatrix(X_vnat_test, feature_names=feat_cols))

yhat = (p_vnat_test >= thr).astype(int)
tn, fp, fn, tp = confusion_matrix(y_vnat_test, yhat).ravel()

prec = tp / (tp + fp + 1e-9)
rec  = tp / (tp + fn + 1e-9)
fpr  = fp / (fp + tn + 1e-9)

print("VNAT TEST @ firewall thr:", {"tn":tn,"fp":fp,"fn":fn,"tp":tp,
                                   "precision":prec,"recall":rec,"fpr":fpr})

VNAT TEST @ firewall thr: {'tn': np.int64(197), 'fp': np.int64(0), 'fn': np.int64(0), 'tp': np.int64(13), 'precision': np.float64(0.9999999999230769), 'recall': np.float64(0.9999999999230769), 'fpr': np.float64(0.0)}


In [4]:
def per_capture_recall(df_split, probs, thr):
    tmp = df_split[["capture_id", "label"]].copy()
    tmp["p"] = np.asarray(probs, dtype=float)
    tmp["yhat"] = (tmp["p"] >= float(thr)).astype(int)

    rows = []
    for cid, g in tmp.groupby("capture_id"):
        pos = g[g["label"] == 1]
        if len(pos) == 0:
            continue
        tp = int((pos["yhat"] == 1).sum())
        fn = int((pos["yhat"] == 0).sum())
        rec = tp / (tp + fn + 1e-9)
        rows.append((str(cid), rec, len(pos)))
    rows.sort(key=lambda x: x[1])
    return rows[:10]

worst_iscx = per_capture_recall(iscx_test_df, p_iscx, thr)
print("Worst 10 ISCX captures (capture_id, recall, #pos_flows):")
for cid, rec, npos in worst_iscx:
    print(cid, rec, npos)

Worst 10 ISCX captures (capture_id, recall, #pos_flows):
vpn_vpn_voipbuster1a.pcap 0.15517241379042807 58
vpn_vpn_ftps_a.pcap 0.7232142857078284 112
vpn_vpn_hangouts_chat1b.pcap 0.9838709677260666 62
vpn_vpn_aim_chat1b.pcap 0.9999999998333333 6
